# Semana 2: GitHub + SQL Avanzado + EDA Completo
**Bootcamp:** Fundamentos de Ingeniería de Datos
**Tema:** EDA (Exploratory Data Analysis) con SQL
**Dataset:** `bootcamp.bronze.properties_bronze` (datos crudos de propiedades inmobiliarias)

**Instrucciones**
1. Usar `%sql` al inicio de cada celda SQL
2. Documentar los hallazgos en comentarios
3. Subir las soluciones a GitHub

> **Importante:** un buen EDA es la base de un buen pipeline. Documentá todo lo que encuentres, ya que los hallazgos del EDA guían las transformaciones en Silver.

---
# 🚀 Setup del entorno en Databricks
> Esto se ejecuta **una sola vez**. Crea el catálogo, los esquemas, el volume, y la tabla `properties_bronze`.

## Paso 1 y 2: Crear catálogo y esquemas (Landing y Bronze)

In [0]:
%sql
-- ============================================================
-- PASO 2: Crear catálogo y esquemas
-- ============================================================

-- Crear catálogo si no existe
CREATE CATALOG IF NOT EXISTS bootcamp;

-- Usar el catálogo
USE CATALOG bootcamp;

-- Esquema Landing (archivos)
CREATE SCHEMA IF NOT EXISTS bootcamp.landing
COMMENT 'Esquema para archivos';

-- Esquema Bronze (datos crudos)
CREATE SCHEMA IF NOT EXISTS bootcamp.bronze
COMMENT 'Esquema para datos crudos sin procesar';

-- Verificar
SHOW SCHEMAS;

## Paso 3: Crear el Volume para archivos

In [0]:
%sql
-- ============================================================
-- PASO 3: Crear el Volume para archivos
-- ============================================================

-- Volume tipo MANAGED (Databricks administra el storage)
CREATE VOLUME IF NOT EXISTS bootcamp.landing.archivos
COMMENT 'Volume para almacenar archivos CSV crudos';

-- Verificar que se creó
SHOW VOLUMES IN bootcamp.landing;

## Paso 4: Subir el archivo CSV
1. En el panel izquierdo de Databricks, hacé clic en **Catalog**.
2. Navegá hasta tu catálogo → `landing` → `archivos`.
3. Hacé clic en **Upload** y arrastrá el archivo `properties_raw.csv`.

📍 Ruta del archivo: `/Volumes/bootcamp/landing/archivos/properties_raw.csv`

In [0]:
%sql
-- ============================================================
-- PASO 4: Verificar que el archivo se subió correctamente
-- ============================================================

LIST '/Volumes/bootcamp/landing/archivos/';

## Paso 5: Crear la tabla `properties_bronze`

In [0]:
%sql
-- Leer directo del archivo es posible, y es útil también para analizarlo
SELECT * FROM read_files(
  '/Volumes/bootcamp/landing/archivos/properties_raw.csv',
  format => 'csv',
  header => true
)
LIMIT 10;

In [0]:
%sql
-- ============================================================
-- PASO 5: Crear la tabla properties_bronze
-- ============================================================

-- Primero eliminamos la tabla si existe (para poder recrearla)
DROP TABLE IF EXISTS bootcamp.bronze.properties_bronze;

-- Creamos la tabla leyendo el CSV, quedándonos solo con registros con url válida
CREATE TABLE bootcamp.bronze.properties_bronze AS
SELECT * FROM read_files(
  '/Volumes/bootcamp/landing/archivos/properties_raw.csv',
  format => 'csv',
  header => true
)
WHERE url LIKE 'https%';  -- Filtramos por links consistentes

In [0]:
%sql
-- Verificar que la tabla se creó correctamente
SHOW TABLES IN bootcamp.bronze;

---
# PARTE 0: Práctica de CTEs y Window Functions
**Dataset:** `samples.nyctaxi.trips` (tabla de ejemplo de Databricks)

Antes del EDA de propiedades, practicamos CTEs y Window Functions con los datos de taxis de NYC de la Semana 1.

**¿Qué son las CTEs?** Consultas temporales definidas con `WITH`. Sirven para reutilizar una subconsulta varias veces, hacer queries más legibles y crear «vistas temporales» dentro de una query.

**¿Qué son las Window Functions?** Cálculos sobre un conjunto de filas relacionadas *sin agrupar* los datos: rankings, comparar filas con promedios/totales, acumulados y tendencias, sin perder el detalle de cada fila.

### Ejercicio 0.1: CTE básica - Estadísticas por zona

**Objetivo:** usar una CTE para organizar cálculos.

**Pregunta:** calculá el promedio de tarifa y distancia por zona de inicio (`pickup_zip`). Usá una CTE para calcular primero las estadísticas por zona, y luego mostrá solo las zonas con más de 100 viajes.

*Pista: WITH estadisticas_por_zona AS (...) luego filtrá con HAVING*

**VERIFICAR: PORQUE HAY ERRORES, MINIMOS DE VIAJE Y FACTURA QUE HAY QUE LIMPIAR EN EL DATASET**

In [0]:
%sql
select min(fare_amount) from samples.nyctaxi.trips;

select min(trip_distance) from samples.nyctaxi.trips;


In [0]:
%sql
-- Tu código acá

WITH estadisticas_por_zona as 
(
    SELECT
    COUNT(*) as total_viajes,
    pickup_zip as zona_inicio,  
    ROUND(AVG(fare_amount),2) as promedio_tarifa,
    ROUND(AVG(trip_distance),2) as promedio_distancia
    from samples.nyctaxi.trips 
    where fare_amount > 0 and trip_distance > 0
    group by pickup_zip
    HAVING total_viajes > 100 -- el having va ocn el group by, no fuera
)
SELECT * FROM estadisticas_por_zona
ORDER BY total_viajes DESC


-- Hallazgos:
-- medio traido de los pelos la CTE

### Ejercicio 0.2: CTE múltiple - Análisis comparativo

**Objetivo:** usar múltiples CTEs para organizar cálculos complejos.

**Pregunta:** comparar el promedio de tarifa por hora del día con el promedio general. Usá dos CTEs: una para el promedio por hora, otra para el promedio general, y luego unilas para mostrar la diferencia.

*Pista: WITH promedio_por_hora AS (...), promedio_general AS (...)*

In [0]:
%sql
-- Tu código acá
WITH promedio_por_hora AS (
    SELECT
        HOUR(tpep_pickup_datetime) AS hora,
        AVG(fare_amount) AS promedio_tarifa_hora
    FROM
        samples.nyctaxi.trips
    WHERE
        fare_amount > 0 
    GROUP BY hora
),
promedio_general AS (
    SELECT
        AVG(fare_amount) AS promedio_gral
    FROM
        samples.nyctaxi.trips
    WHERE
        fare_amount > 0   
)
SELECT
promedio_por_hora.hora,
    ROUND(promedio_por_hora.promedio_tarifa_hora,2) AS promedio_tarifa_hora,
    ROUND(promedio_general.promedio_gral, 2) as promedio_gral,
    ROUND(promedio_por_hora.promedio_tarifa_hora - promedio_general.promedio_gral,2) as diferencia
    FROM
        promedio_por_hora
    CROSS JOIN
        promedio_general
    ORDER BY hora;
-- Hallazgos:
-- distintas métricas en cada CTE, luego unirlas en el SELECT FINAL

### Ejercicio 0.3: CTE para limpieza de datos

**Objetivo:** usar una CTE para filtrar datos antes de analizar.

**Pregunta:** calculá estadísticas de distancia solo para viajes válidos (distancia > 0, tarifa > 0). Usá una CTE para filtrar primero los datos válidos, y luego calculá mínimo, máximo, promedio y mediana.

*Pista: WITH viajes_validos AS (SELECT ... WHERE ...)*

In [0]:
%sql
-- Tu código acá
WITH datos_limpios AS 
(
    SELECT
    *
    FROM samples.nyctaxi.trips
    WHERE
    fare_amount > 0 AND trip_distance > 0
)
    SELECT 
    MIN(trip_distance) as distancia_minima,
    MAX(trip_distance) as distancia_maxima,
    AVG(trip_distance) as promedio_distancia,
    MEDIAN(trip_distance) as mediana_distancia
    FROM datos_limpios
-- Hallazgos:
-- 

### Ejercicio 0.4: Window Function - ROW_NUMBER() para ranking

**Objetivo:** usar `ROW_NUMBER()` para crear rankings.

**Pregunta:** creá un ranking de los viajes más caros. Mostrá los top 20 viajes con su fecha, distancia, tarifa y el ranking (1 = más caro).

*Pista: ROW_NUMBER() OVER (ORDER BY fare_amount DESC)*

In [0]:
%sql
-- Tu código acá
SELECT
ROW_NUMBER() OVER (ORDER BY fare_amount DESC) as rnk,
fare_amount,
trip_distance,
tpep_pickup_datetime
FROM samples.nyctaxi.trips
ORDER BY fare_amount DESC


-- Hallazgos:
-- el order by en ROW_NUMBER nos ayuda a definir QUE COLUMNA usar el RANKING y si rankear de manera asc o desc

### Ejercicio 0.5: ROW_NUMBER() vs RANK() vs DENSE_RANK()

**Objetivo:** comparar las 3 funciones de ranking sobre el mismo dataset y entender cómo manejan los empates.

**Pregunta:** usando la misma tabla de viajes, mostrá los top 20 viajes más caros con su fecha, tarifa y tres columnas de ranking: `ROW_NUMBER()`, `RANK()` y `DENSE_RANK()`. ¿Qué pasa cuando hay tarifas repetidas?

*Pista: usá las 3 funciones con OVER (ORDER BY fare_amount DESC) en el mismo SELECT*

In [0]:
%sql
-- Tu código acá
SELECT
  tpep_pickup_datetime,
  fare_amount,
 ROW_NUMBER() OVER (ORDER BY fare_amount DESC) AS rn,
 RANK() OVER (ORDER BY fare_amount DESC) AS rk,
 DENSE_RANK() OVER (ORDER BY fare_amount DESC) AS drk
FROM samples.nyctaxi.trips
ORDER BY fare_amount DESC
limit 20;

-- Hallazgos:
-- 

### DEFINICIONES DE CADA WINDOW FUNCTION: 


![image_1793211929281.png](./image_1793211929281.png "image_1793211929281.png")

**ROW_NUMBER()** numera cada fila con un número único, sin repetir nunca. Si hay empate, desempata de forma arbitraria, así que a cuál de las dos filas de 45 le toca el 2 y a cuál el 3 puede cambiar entre ejecuciones. _Para que sea determinista tenés que agregar más columnas al ORDER BY_.

**RANK()** les da el mismo número a los empatados y después saltea posiciones. Hubo dos en el puesto 2, entonces no existe el 3 y sigue el 4. Es como un podio olímpico: dos platas, no hay bronce.

**DENSE_RANK()** les da el mismo número a los empatados pero no saltea: después del 2 viene el 3. Cuenta valores distintos, no filas.

**Cuándo usar cada una:**

- ROW_NUMBER(): cuando necesitás exactamente una fila por grupo, por ejemplo «el último registro de cada cliente» o quedarte con una fila al deduplicar. Es la que más vas a usar en pipelines.
- RANK(): rankings donde los saltos tienen sentido, como una posición en una competencia.
- DENSE_RANK(): cuando querés «el 2º precio más alto» o «los 3 niveles de tarifa más caros» sin que los empates te corran la numeración.

## Ejercicios 0.6 - 0.10: Window Functions avanzadas

### Ejercicio 0.6: Comparar con promedio general

**Objetivo:** usar `AVG() OVER()` para comparar con el promedio.

**Pregunta:** para **cada viaje** (esta es la clave la GARNULARIDAD), mostrá la tarifa, la distancia, y cómo se compara la tarifa con el promedio general (diferencia y porcentaje de diferencia).

*Pista: AVG(fare_amount) OVER() calcula el promedio de todas las filas*

Ejemplo del video: 

![image_1793212146294.png](./image_1793212146294.png "image_1793212146294.png") 

In [0]:
%sql
-- Tu código acá
SELECT
tpep_pickup_datetime, --PICKUP_ZIP NO DEFINE CADA VIAJE, porque ví que se repetían
trip_distance,
fare_amount as tarifa,
AVG(fare_amount) OVER() as promedio_global_tarifa, --SIN NADA EN EL OVER ES SIN PARTICIONAR POR UN CAMPO CATEGÓRICO
promedio_global_tarifa - fare_amount as diferencia_con_promedio,
ROUND((promedio_global_tarifa - fare_amount) / promedio_global_tarifa,2) as pct_dif
FROM samples.nyctaxi.trips;

-- Hallazgos:
-- 

### Ejercicio 0.7: Promedio por partición

**Objetivo:** usar `PARTITION BY` para calcular promedios por grupo.

**Pregunta:** para cada viaje, mostrá: fecha, tarifa, zona de inicio (`pickup_zip`), y el promedio de tarifa para esa misma zona (ej: si es zona 10001, el promedio de todos los viajes de la zona 10001).

*Pista: AVG(fare_amount) OVER (PARTITION BY pickup_zip)*

In [0]:
%sql
-- Tu código acá
SELECT
tpep_pickup_datetime, --PICKUP_ZIP NO DEFINE CADA VIAJE, porque ví que se repetían
fare_amount as tarifa,
pickup_zip as zona_inicio,
ROUND(AVG(fare_amount) OVER(PARTITION BY pickup_zip),2) as promedio_global_zona 
FROM samples.nyctaxi.trips;

-- Hallazgos:
-- 

### Ejercicio 0.8: LAG() para comparar con anterior

**Objetivo:** usar `LAG()` para comparar con el registro anterior.

**Pregunta:** ordená los viajes por fecha y mostrá: fecha, tarifa, y la tarifa del viaje anterior (si existe). Esto permite ver si hay tendencias temporales.

*Pista: LAG(fare_amount) OVER (ORDER BY tpep_pickup_datetime)*

In [0]:
%sql
-- Tu código acá

SELECT
tpep_pickup_datetime, 
fare_amount as tarifa,
ROUND(LAG(fare_amount) OVER(ORDER BY tpep_pickup_datetime),2) as tarifa_anterior 
--ACA YA NO VA UNA PARTICION SINO UN ORDENAMIENTO INTERNO
FROM samples.nyctaxi.trips
ORDER BY tpep_pickup_datetime, tarifa
-- Hallazgos:
-- 

### Ejercicio 0.9: SUM() acumulado

**Objetivo:** usar `SUM() OVER()` para calcular acumulados.

**Pregunta:** ordená los viajes por fecha y mostrá: fecha, tarifa, y el total acumulado de tarifas hasta esa fecha (running total).

*Pista: SUM(fare_amount) OVER (ORDER BY tpep_pickup_datetime ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)*

**⚠️SIRVE PARA MEDIAS MOVILES**

In [0]:
%sql
-- Tu código acá
SELECT 
tpep_pickup_datetime,
fare_amount,
SUM(fare_amount) OVER 
(ORDER BY tpep_pickup_datetime ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) as total_acumulado
FROM samples.nyctaxi.trips

-- Hallazgos:
-- 

### Ejercicio 0.10: Combinando CTEs y Window Functions

**Objetivo:** combinar ambas técnicas en una query compleja.

**Pregunta:** usá CTEs para:
1. Filtrar viajes válidos (distancia > 0, tarifa > 0)
2. Calcular estadísticas por zona (promedio, máximo, mínimo)
3. Luego usá Window Functions para rankear las zonas por promedio de tarifa
4. Mostrar el top 10 zonas más caras con su ranking

*Pista: combiná WITH para las CTEs y ROW_NUMBER() OVER() para el ranking*

In [0]:
%sql
-- Tu código acá
with estadisticas as (
SELECT 
    pickup_zip as zona,
    ROUND(AVG(fare_amount),2) as promedio_tarifa,   
    ROUND(MAX(fare_amount),2) as tarifa_maxima,
    ROUND(MIN(fare_amount),2) as tarifa_minima 
FROM samples.nyctaxi.trips
WHERE trip_distance > 0 and fare_amount > 0
GROUP BY pickup_zip  
)
select 
*,
ROW_NUMBER() OVER (ORDER BY promedio_tarifa DESC) as ranking
FROM estadisticas
LIMIT 10
-- Hallazgos:
-- 